# Notebook 03 – Machine Learning Models
**DSA 210 – Screen Time & Well-Being Project**

Two ML tasks:
1. **Regression** – Predict next-day total screen time
2. **Classification** – Predict whether tomorrow will be a "high screen time" day (above personal median)

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import cross_val_score, LeaveOneOut, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.inspection import permutation_importance

import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=1.1)

RANDOM_STATE = 42

In [ ]:
df = pd.read_csv('../data/processed/merged_dataset.csv', parse_dates=['date'])

# Target for regression: next-day screen time
df['next_day_screen_time'] = df['total_screen_time_min'].shift(-1)

FEATURE_COLS = [
    'total_screen_time_min', 'social_min', 'entertainment_min', 'productivity_min',
    'pickups', 'notifications', 'mood_score', 'productivity_score',
    'sleep_hours', 'exercise_min', 'workload_encoded', 'is_weekend',
    'social_ratio', 'prev_day_mood'
]

# Drop rows with NaN in any feature or target
reg_df = df[FEATURE_COLS + ['next_day_screen_time']].dropna()
cls_df = df[FEATURE_COLS + ['high_screen_time']].dropna()

print(f'Regression dataset: {reg_df.shape}')
print(f'Classification dataset: {cls_df.shape}')
print(f'Class balance (high=1): {cls_df["high_screen_time"].value_counts().to_dict()}')

---
## Part 1: Regression – Predicting Next-Day Screen Time

In [ ]:
X_reg = reg_df[FEATURE_COLS].values
y_reg = reg_df['next_day_screen_time'].values

# Models to compare
reg_models = {
    'Linear Regression': Pipeline([('scaler', StandardScaler()), ('model', LinearRegression())]),
    'Ridge Regression':  Pipeline([('scaler', StandardScaler()), ('model', Ridge(alpha=1.0))]),
    'Random Forest':     RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE),
}

# Leave-One-Out CV (good for small datasets)
loo = LeaveOneOut()
reg_results = {}

for name, model in reg_models.items():
    cv_scores = cross_val_score(model, X_reg, y_reg, cv=loo, scoring='neg_mean_absolute_error')
    mae = -cv_scores.mean()
    cv_r2 = cross_val_score(model, X_reg, y_reg, cv=5, scoring='r2').mean()
    reg_results[name] = {'LOO-MAE': mae, '5-Fold R²': cv_r2}
    print(f'{name:25s} → LOO-MAE: {mae:.1f} min | 5-Fold R²: {cv_r2:.3f}')

In [ ]:
# Fit best model (Random Forest) and inspect feature importance
best_reg = RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE)
best_reg.fit(X_reg, y_reg)

feat_imp = pd.Series(best_reg.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 6))
feat_imp.plot(kind='barh', color='steelblue', ax=ax)
ax.set_title('Random Forest – Feature Importance (Regression)')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig('../figures/reg_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Actual vs Predicted plot (training fit for visualization)
y_pred_reg = best_reg.predict(X_reg)

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(y_reg, y_pred_reg, alpha=0.6, color='steelblue')
lims = [min(y_reg.min(), y_pred_reg.min()), max(y_reg.max(), y_pred_reg.max())]
ax.plot(lims, lims, 'r--', linewidth=1.5, label='Perfect prediction')
ax.set_xlabel('Actual Next-Day Screen Time (min)')
ax.set_ylabel('Predicted (min)')
ax.set_title('Actual vs Predicted – Random Forest Regression')
ax.legend()
plt.tight_layout()
plt.savefig('../figures/reg_actual_vs_pred.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Part 2: Classification – High vs Low Screen Time Day

In [ ]:
X_cls = cls_df[FEATURE_COLS].values
y_cls = cls_df['high_screen_time'].values

cls_models = {
    'Logistic Regression':     Pipeline([('scaler', StandardScaler()), ('model', LogisticRegression(max_iter=1000))]),
    'Random Forest Classifier': RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE),
    'Gradient Boosting':        GradientBoostingClassifier(n_estimators=100, random_state=RANDOM_STATE),
}

cls_results = {}
for name, model in cls_models.items():
    scores = cross_val_score(model, X_cls, y_cls, cv=5, scoring='f1')
    acc    = cross_val_score(model, X_cls, y_cls, cv=5, scoring='accuracy').mean()
    cls_results[name] = {'5-Fold F1': scores.mean(), 'Accuracy': acc}
    print(f'{name:30s} → F1: {scores.mean():.3f} (±{scores.std():.3f}) | Accuracy: {acc:.3f}')

In [ ]:
# Confusion matrix for Random Forest
from sklearn.model_selection import StratifiedKFold

best_cls = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE)
best_cls.fit(X_cls, y_cls)

y_pred_cls = best_cls.predict(X_cls)
print(classification_report(y_cls, y_pred_cls, target_names=['Low Screen Time', 'High Screen Time']))

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_estimator(
    best_cls, X_cls, y_cls, display_labels=['Low', 'High'],
    colorbar=False, ax=ax
)
ax.set_title('Confusion Matrix – Random Forest Classifier')
plt.tight_layout()
plt.savefig('../figures/cls_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Feature importance for classification
feat_imp_cls = pd.Series(best_cls.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 6))
feat_imp_cls.plot(kind='barh', color='coral', ax=ax)
ax.set_title('Random Forest – Feature Importance (Classification)')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig('../figures/cls_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Model Comparison Summary

In [ ]:
print('=== REGRESSION RESULTS ===')
pd.DataFrame(reg_results).T.round(3)

In [ ]:
print('=== CLASSIFICATION RESULTS ===')
pd.DataFrame(cls_results).T.round(3)